In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy


In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/"
# data = sc.read_h5ad(data_annotated_path + "joint_annotated.h5ad")
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden.h5ad")

data.shape

# Building A Timeline

In [ ]:
plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [data.obs['log_total_counts'][data.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()


In [ ]:
# PCA IQR Timeline, Big Sheet

In [ ]:
control_mask =  data.obs['exp_condition'] == "Cntr"
mutant_mask = data.obs['exp_condition'] == "Mtz"

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

fig,axes = plt.subplots(5,10,figsize=(30,15))
for i in range(50):

    time_control_iqrs = {}
    time_mutant_iqrs = {}
    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask
        time_control_iqrs[t] = {
            'low':np.percentile(data.obsm['X_pca'][:,i][combined_control_mask],25),
            'median':np.median(data.obsm['X_pca'][:,i][combined_control_mask]),
            'high':np.percentile(data.obsm['X_pca'][:,i][combined_control_mask],75)
        }
        time_mutant_iqrs[t] = {
            'low':np.percentile(data.obsm['X_pca'][:,i][combined_mutant_mask],25),
            'median':np.median(data.obsm['X_pca'][:,i][combined_mutant_mask]),
            'high':np.percentile(data.obsm['X_pca'][:,i][combined_mutant_mask],75)
        }
    
    ax = axes[i//10,i%10]
    ax.plot(
        times,
        [time_control_iqrs[t]['median'] for t in times],
        label="control"
    )
    ax.fill_between(
        times,
        [time_control_iqrs[t]['low'] for t in times],
        [time_control_iqrs[t]['high'] for t in times],
        alpha=.2
    )
    ax.plot(
        times,
        [time_mutant_iqrs[t]['median'] for t in times],
        label="mutant"
    )
    ax.fill_between(
        times,
        [time_mutant_iqrs[t]['low'] for t in times],
        [time_mutant_iqrs[t]['high'] for t in times],
        alpha=.2
    )
    ax.set_title(f"PC{i+1}")
    ax.legend()
    # ax.plot(
    #     control_data.obs['time'],
    #     control_data.obsm['X_pca'][:,i],
    #     label="control"
    # )
    # ax.plot(
    #     mutant_data.obs['time'],
    #     mutant_data.obsm['X_pca'][:,i],
    #     label="mutant"
    # )
    # ax.legend()
fig.tight_layout()
fig.show()

# Cell Death Scoring

In [ ]:
sc.tl.score_genes(data,data.var_names[data.var['parthanatos']],score_name="parthanatos")
sc.tl.score_genes(data,data.var_names[data.var['necroptosis']],score_name="necroptosis")
sc.tl.score_genes(data,data.var_names[data.var['apoptosis']],score_name="apoptosis")


In [ ]:
# Let's plot a timecourse for each between mutant and ctrl 

times = [12,24,48,72]

time_masks = [
    data.obs['time'] == t
    for t in times
]

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_iqrs = {}
    time_mutant_iqrs = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask
        time_control_iqrs[t] = {
            'low':np.percentile(data.obs[score][combined_control_mask],25),
            'median':np.median(data.obs[score][combined_control_mask]),
            'high':np.percentile(data.obs[score][combined_control_mask],75)
        }
        time_mutant_iqrs[t] = {
            'low':np.percentile(data.obs[score][combined_mutant_mask],25),
            'median':np.median(data.obs[score][combined_mutant_mask]),
            'high':np.percentile(data.obs[score][combined_mutant_mask],75)
        }


    ax = axes[i]
    ax.plot(
        times,
        [time_control_iqrs[t]['median'] for t in times],
        label="control (median)"
    )
    ax.fill_between(
        times,
        [time_control_iqrs[t]['low'] for t in times],
        [time_control_iqrs[t]['high'] for t in times],
        alpha=.2,label="control (IQR)"
    )
    ax.plot(
        times,
        [time_mutant_iqrs[t]['median'] for t in times],
        label="mutant (median)"
    )
    ax.fill_between(
        times,
        [time_mutant_iqrs[t]['low'] for t in times],
        [time_mutant_iqrs[t]['high'] for t in times],
        alpha=.2,label="mutant (IQR)"
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Naive Score")
    ax.set_title(f"{score}")
    ax.legend(loc='upper left')
fig.tight_layout()
fig.show()



fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_sem_range = {}
    time_mutant_sem_range = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask

        time_control_sem = scipy.stats.sem(data.obs[score][combined_control_mask])
        time_control_mean = np.mean(data.obs[score][combined_control_mask])
        time_control_sem_range[t] = {
            'low':time_control_mean - (time_control_sem*2),
            'mean':time_control_mean,
            'high':time_control_mean + (time_control_sem*2)
        }
        time_mutant_sem = scipy.stats.sem(data.obs[score][combined_mutant_mask])
        time_mutant_mean = np.mean(data.obs[score][combined_mutant_mask])
        time_mutant_sem_range[t] = {
            'low':time_mutant_mean - (time_mutant_sem*2),
            'mean':time_mutant_mean,
            'high':time_mutant_mean + (time_mutant_sem*2)
        }

    ax = axes[i]

    ax.plot(
        times,
        [time_control_sem_range[t]['mean'] for t in times],
        label="control (mean)"
    )
    ax.fill_between(
        times,
        [time_control_sem_range[t]['low'] for t in times],
        [time_control_sem_range[t]['high'] for t in times],
        alpha=.2,label="control (SEM)"
    )
    ax.plot(
        times,
        [time_mutant_sem_range[t]['mean'] for t in times],
        label="mutant (mean)"
    )
    ax.fill_between(
        times,
        [time_mutant_sem_range[t]['low'] for t in times],
        [time_mutant_sem_range[t]['high'] for t in times],
        alpha=.2,label="mutant (SEM)"
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(f"{score}")
    ax.set_title(f"{score} (All Cells)")
    ax.legend()
    
fig.tight_layout()
fig.show()

In [ ]:
np.array(
    [np.around(time_control_sem_range[t]['mean'],3) for t in times]
) - np.array(
    [np.around(time_mutant_sem_range[t]['mean'],3) for t in times]
)

In [ ]:
print([time_control_sem_range[t]['mean'] for t in times])

# Leiden 

In [ ]:
# sc.tl.leiden(data, resolution=0.5)

In [ ]:
# sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
sc.tl.rank_genes_groups(data, 'leiden')
sc.pl.rank_genes_groups(data, n_genes=25)


In [ ]:
top_3_across_all = data.uns['rank_genes_groups']['names'][:3]
top_3_across_all = np.array([[top_3_across_all[j][i] for i in range(20)] for j in range(3)])

In [ ]:
top_3_across_all.flatten()

In [ ]:
sc.pl.dotplot(data, top_3_across_all.flatten(),groupby="leiden",figsize=(20,10))
sc.pl.dotplot(data, ['rbpms2a','rbpms2b'],groupby="leiden",figsize=(20,10))

In [ ]:
sc.write("full_annotations_leiden.h5ad",data)

# Subset to Radial Glia

In [ ]:
rgc_mask = (data.obs['leiden'] == '3') | (data.obs['leiden'] == '10')
rgc = data[rgc_mask]
rgc

In [ ]:
# Let's plot a timecourse for each between mutant and ctrl 

times = [12,24,48,72]

time_masks = [
    rgc.obs['time'] == t
    for t in times
]

control_mask =  rgc.obs['exp_condition'] == "Cntr"
mutant_mask = rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_iqrs = {}
    time_mutant_iqrs = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask
        time_control_iqrs[t] = {
            'low':np.percentile(rgc.obs[score][combined_control_mask],25),
            'median':np.median(rgc.obs[score][combined_control_mask]),
            'high':np.percentile(rgc.obs[score][combined_control_mask],75)
        }
        time_mutant_iqrs[t] = {
            'low':np.percentile(rgc.obs[score][combined_mutant_mask],25),
            'median':np.median(rgc.obs[score][combined_mutant_mask]),
            'high':np.percentile(rgc.obs[score][combined_mutant_mask],75)
        }


    ax = axes[i]
    ax.plot(
        times,
        [time_control_iqrs[t]['median'] for t in times],
        label="control (median)"
    )
    ax.fill_between(
        times,
        [time_control_iqrs[t]['low'] for t in times],
        [time_control_iqrs[t]['high'] for t in times],
        alpha=.2,label="control (IQR)"
    )
    ax.plot(
        times,
        [time_mutant_iqrs[t]['median'] for t in times],
        label="mutant (median)"
    )
    ax.fill_between(
        times,
        [time_mutant_iqrs[t]['low'] for t in times],
        [time_mutant_iqrs[t]['high'] for t in times],
        alpha=.2,label="mutant (IQR)"
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("Naive Score")
    ax.set_title(f"{score}, (Radial Glia)")
    ax.legend()
fig.tight_layout()
fig.show()



fig,axes = plt.subplots(1,3,figsize=(15,5))
for i,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_control_sem_range = {}
    time_mutant_sem_range = {}

    for t,time_mask in zip(times,time_masks):
        combined_control_mask = control_mask & time_mask
        combined_mutant_mask = mutant_mask & time_mask

        time_control_sem = scipy.stats.sem(rgc.obs[score][combined_control_mask])
        time_control_mean = np.mean(rgc.obs[score][combined_control_mask])
        time_control_sem_range[t] = {
            'low':time_control_mean - (time_control_sem*2),
            'mean':time_control_mean,
            'high':time_control_mean + (time_control_sem*2)
        }
        time_mutant_sem = scipy.stats.sem(rgc.obs[score][combined_mutant_mask])
        time_mutant_mean = np.mean(rgc.obs[score][combined_mutant_mask])
        time_mutant_sem_range[t] = {
            'low':time_mutant_mean - (time_mutant_sem*2),
            'mean':time_mutant_mean,
            'high':time_mutant_mean + (time_mutant_sem*2)
        }

    ax = axes[i]

    ax.plot(
        times,
        [time_control_sem_range[t]['mean'] for t in times],
        label="control (mean)"
    )
    ax.fill_between(
        times,
        [time_control_sem_range[t]['low'] for t in times],
        [time_control_sem_range[t]['high'] for t in times],
        alpha=.2,label="control (SEM)"
    )
    ax.plot(
        times,
        [time_mutant_sem_range[t]['mean'] for t in times],
        label="mutant (mean)"
    )
    ax.fill_between(
        times,
        [time_mutant_sem_range[t]['low'] for t in times],
        [time_mutant_sem_range[t]['high'] for t in times],
        alpha=.2,label="mutant (SEM)"
    )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(f"{score}")
    ax.set_title(f"{score} (Radial Glia)")
    ax.legend()
    
fig.tight_layout()
fig.show()

# RGC Vs Rest Comparison

In [ ]:
non_rgc = data[~rgc_mask]

times = [12,24,48,72]

rgc_time_masks = [
    rgc.obs['time'] == t
    for t in times
]

non_rgc_time_masks = [
    non_rgc.obs['time'] == t
    for t in times
]

rgc_control_mask =  rgc.obs['exp_condition'] == "Cntr"
rgc_mutant_mask = rgc.obs['exp_condition'] == "Mtz"

non_rgc_control_mask =  non_rgc.obs['exp_condition'] == "Cntr"
non_rgc_mutant_mask = non_rgc.obs['exp_condition'] == "Mtz"

fig,axes = plt.subplots(1,3,figsize=(15,5))
for j,score in enumerate(['parthanatos','necroptosis','apoptosis']):

    time_rgc_control_iqrs = {}
    time_rgc_mutant_iqrs = {}
    time_non_rgc_control_iqrs = {}
    time_non_rgc_mutant_iqrs = {}

    for i,t in enumerate(times):

        combined_rgc_control_mask = rgc_control_mask & rgc_time_masks[i]
        combined_rgc_mutant_mask = rgc_mutant_mask & rgc_time_masks[i]

        combined_non_rgc_control_mask = non_rgc_control_mask & non_rgc_time_masks[i]
        combined_non_rgc_mutant_mask = non_rgc_mutant_mask & non_rgc_time_masks[i]

        time_rgc_control_iqrs[t] = {
            'low':np.percentile(rgc.obs[score][combined_rgc_control_mask],25),
            'median':np.median(rgc.obs[score][combined_rgc_control_mask]),
            'high':np.percentile(rgc.obs[score][combined_rgc_control_mask],75)
        }
        time_rgc_mutant_iqrs[t] = {
            'low':np.percentile(rgc.obs[score][combined_rgc_mutant_mask],25),
            'median':np.median(rgc.obs[score][combined_rgc_mutant_mask]),
            'high':np.percentile(rgc.obs[score][combined_rgc_mutant_mask],75)
        }

        time_non_rgc_control_iqrs[t] = {
            'low':np.percentile(non_rgc.obs[score][combined_non_rgc_control_mask],25),
            'median':np.median(non_rgc.obs[score][combined_non_rgc_control_mask]),
            'high':np.percentile(non_rgc.obs[score][combined_non_rgc_control_mask],75)
        }

        time_non_rgc_mutant_iqrs[t] = {
            'low':np.percentile(non_rgc.obs[score][combined_non_rgc_mutant_mask],25),
            'median':np.median(non_rgc.obs[score][combined_non_rgc_mutant_mask]),
            'high':np.percentile(non_rgc.obs[score][combined_non_rgc_mutant_mask],75)
        }

    def filled_centerline_to_ax(ax,times,iqrs,label,style,**kwargs):
        ax.plot(
            times,
            [iqrs[t]['median'] for t in times],
            style,
            label=label,
            **kwargs
        )
        # ax.fill_between(
        #     times,
        #     [iqrs[t]['low'] for t in times],
        #     [iqrs[t]['high'] for t in times],
        #     alpha=.2,label=label
        # )
        
        ax.set_xlabel("Time (h)")
        ax.set_ylabel(f"{score}")
        ax.set_title(f"{score}")
        ax.legend()
        
    ax = axes[j]

    filled_centerline_to_ax(ax,times,time_rgc_control_iqrs,"RG Control",style='r')
    filled_centerline_to_ax(ax,times,time_rgc_mutant_iqrs,"RG Mutant",style='b')
    filled_centerline_to_ax(ax,times,time_non_rgc_control_iqrs,"Non-RG Control",style='r--',alpha=.3)
    filled_centerline_to_ax(ax,times,time_non_rgc_mutant_iqrs,"Non-RG Mutant",style='b--',alpha=.3)

    ax.legend()
fig.tight_layout()
fig.show()


In [ ]:
rgc_time_masks